### Set up environment

In [1]:
%pip install google-cloud-aiplatform python-dotenv fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### author

In [7]:
!gcloud auth login

!gcloud auth application-default login

'gcloud' is not recognized as an internal or external command,
operable program or batch file.
'gcloud' is not recognized as an internal or external command,
operable program or batch file.


### Deep learning container

In [ ]:
import os
from google.cloud import aiplatform

aiplatform.init(
    project=os.environ.get("GCP_PROJECT_ID","ticmastery"),
    location=os.environ.get("GCP_REGION","us-central1")
)

model = aiplatform.Model.upload(
    display_name="hf-ticmastery",
    serving_container_image_uri="us-docker.pkg.dev/deeplearning-platform-release/gcs-fuse/huggingface-text-generation-inference.1-3.py310",
    serving_container_environment_variables={
        "HF_MODEL_TO": "Qwen/Qwen2.5-0.5B-Instruct",
        "HF_TASK": "text-generation",
        "MAX_INPUT_LENGTH": "1024",
        "MAX_TOTAL_TOKENS": "2048"
    }
)

endpoint = aiplatform.Endpoint.create(display_name="hf-ticmastery-endpoint")

model.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1
)

### Custom Serving Container

In [ ]:
# Create Artifact Registry Repository for Custom Model
!gcloud artifacts repositories create custom-model-qwen \
    --repository-format=docker \
    --location=us-central1 \
    --description="Custom Model Repository for Qwen"

# Configure Docker to use Artifact Registry
!gcloud auth configure-docker us-central1-docker.pkg.dev

# Build Docker image
!docker build -t custom-model-qwen:latest .

# Tag Docker image
!docker tag custom-model-qwen:latest us-central1-docker.pkg.dev/ticmastery/custom-model-qwen:v1
# Push Docker image to Artifact Registry
!docker push us-central1-docker.pkg.dev/ticmastery/custom-model-qwen:v1

In [ ]:
# download weight to GCS
!gcloud storage cp -r ./notebook/qwen-2.5b-finetuned-qlora gs://ticmastery/qwen-2.5b-finetuned-qlora

import os
from google.cloud import aiplatform

aiplatform.init(
    project=os.environ.get("GCP_PROJECT_ID", "ticmastery"),
    location=os.environ.get("GCP_REGION", "us-central1")
)

model  = aiplatform.Model.upload(
    display_name="hf-ticmastery-custom",
    artifact_uri="gs://ticmastery/qwen-2.5b-finetuned-qlora",
    serving_container_image_uri="us-central1-docker.pkg.dev/ticmastery/custom-model-qwen:v1",
    serving_container_ports=[8080]
)

endpoint = aiplatform.Endpoint.create(display_name="hf-ticmastery-custom-endpoint")

model.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1
)

### Test

In [ ]:
from google.cloud import aiplatform

ENDPOINT_ID = endpoint.resource_name
print(f"Using deployed Endpoint ID: {ENDPOINT_ID}")

# ENDPOINT_ID = "projects/ticmastery/locations/us-central1/endpoints/[YOUR_ENDPOINT_NUMERIC_ID]"
# print(f"Using manual Endpoint ID: {ENDPOINT_ID}")

endpoint_test = aiplatform.Endpoint(endpoint_name=ENDPOINT_ID)

response = endpoint_test.predict(
    instances=["What is the capital of France?"]
)
print(response.predictions)

In [ ]:
from google.cloud import aiplatform

endpoint.undeploy_all()
endpoint.delete()
print("Successfully undeployed and deleted the active endpoint.")

# ENDPOINT_ID = "projects/ticmastery/locations/us-central1/endpoints/[YOUR_ENDPOINT_NUMERIC_ID]"
# print(f"Deleting manual Endpoint ID: {ENDPOINT_ID}")
# endpoint_to_delete = aiplatform.Endpoint(endpoint_name=ENDPOINT_ID)
# endpoint_to_delete.undeploy_all()
# endpoint_to_delete.delete()